## 03 · Anotación masiva y adjudicación

**Guía de ejecución:** Entrada: data/processed/conversation_dataset.pkl y prompts. Contiene peticiones Batch a un servicio externo y celdas para recuperar resultados históricos. No ejecutar íntegramente sin revisar costes, identificadores de lote y estado.

Instala las dependencias una vez desde la raíz del repositorio con `pip install -e ".[notebooks]"`. Los archivos de `data/` se mantienen fuera de Git. Este notebook se conserva para reproducir la fase de investigación; no se recomienda pulsar «Ejecutar todo» sin revisar sus fases y requisitos.


In [ ]:
# Preparación: iniciar Jupyter desde la raíz del repositorio (local o Colab).
from pathlib import Path
import os, sys

def _locate_repo():
    for folder in (Path.cwd(), *Path.cwd().parents, Path('/content/cognitive-bias-detector')):
        if (folder / 'pyproject.toml').is_file() and (folder / 'src' / 'bias_detector').is_dir():
            return folder.resolve()
    raise RuntimeError('No se encontró la raíz del repositorio. Abre Jupyter allí o usa %cd /content/cognitive-bias-detector en Colab.')

PROJECT_ROOT = _locate_repo()
source_path = str(PROJECT_ROOT / 'src')
if source_path not in sys.path:
    sys.path.insert(0, source_path)
from bias_detector.config import (
    DATA_DIR, MODEL_DATA_DIR, MODEL_DIR, PROMPTS_DIR, RESULTS_DIR,
    RAW_DIR, PROCESSED_DIR, MODELING_DIR, PILOT_DIR,
    REDACTION_ANALYSIS_DIR, ANNOTATION_RESULTS_DIR,
    MASS_ANNOTATION_DIR, FINAL_ANNOTATION_DIR, PILOT_RESULTS_DIR,
    MICRO_PILOT_RESULTS_DIR, CORPUS_DATA, FINAL_ANNOTATIONS_DATA,
    require_file, ensure_output_dir,
)
print('Repositorio:', PROJECT_ROOT)
print('Directorio de datos:', DATA_DIR)


# Anotación masiva del corpus

### Imports y rutas

In [ ]:
from pathlib import Path
from enum import Enum
from pydantic import BaseModel, Field, ConfigDict
from openai import OpenAI

import numpy as np
import pandas as pd
import json
import os

In [ ]:
PROJECT_ROOT = _locate_repo()

DATA_DIR = PROJECT_ROOT / "data"

PROMPT_DIR = PROJECT_ROOT / "prompts"

MASS_RESULTS_DIR = (
    DATA_DIR
    / "results"
    / "mass_annotation"
)

LUNA_RESULTS_DIR = (
    MASS_RESULTS_DIR
    / "LUNA"
)

LUNA_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

TERRA_RESULTS_DIR = (
    MASS_RESULTS_DIR
    / "terra"
)

TERRA_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

BATCH_INPUT_DIR = (
    LUNA_RESULTS_DIR
    / "batch_inputs"
)

BATCH_INPUT_DIR.mkdir(parents=True, exist_ok=True)

BATCH_OUTPUT_DIR = (
    LUNA_RESULTS_DIR
    / "batch_outputs"
)

BATCH_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

### Carga de datos

In [ ]:
CORPUS_PATH = (
    DATA_DIR
    / "processed"
    / "conversation_dataset.pkl"
)

corpus_df = pd.read_pickle(
    CORPUS_PATH
)

In [ ]:
# Identificadores estables. Idempotente al ejecutar esta celda de nuevo.
corpus_df = corpus_df.copy()
if "conversation_id" not in corpus_df.columns:
    corpus_df.insert(0, "conversation_id", [f"CORPUS_{i:05d}" for i in range(1, len(corpus_df) + 1)])
else:
    if corpus_df["conversation_id"].duplicated().any():
        raise ValueError("Identificadores de conversación duplicados")
print("IDs de conversación:", corpus_df["conversation_id"].nunique())


In [ ]:
CORPUS_FINAL_PATH = (
    DATA_DIR
    / "processed"
    / "mass_corpus_clean.pkl"
)

corpus_df.to_pickle(
    CORPUS_FINAL_PATH
)

### Configuración anotación

In [ ]:
PROMPT_VERSION = "v0.2"

LUNA_MODEL = "gpt-5.6-luna"

BIASES = [
    "confirmation_bias",
    "anchoring_bias",
    "framing_effect",
    "availability_bias"
]

TOTAL_REQUESTS = (
    len(corpus_df)
    * len(BIASES)
)

print(
    "Total de anotaciones LUNA:",
    TOTAL_REQUESTS
)

assert TOTAL_REQUESTS == 38716

In [ ]:
# Esquema estructurado
class AnnotationLabel(str, Enum):
    PRESENT = "PRESENT"
    ABSENT = "ABSENT"
    UNCERTAIN = "UNCERTAIN"


class ConfidenceLevel(str, Enum):
    HIGH = "HIGH"
    MEDIUM = "MEDIUM"
    LOW = "LOW"


class BiasAnnotation(BaseModel):
    model_config = ConfigDict(extra="forbid")
    bias: str
    label: AnnotationLabel
    trigger_turns: list[int]
    evidence_turns: list[int]
    justification: str
    confidence: ConfidenceLevel

In [ ]:
ANNOTATION_SCHEMA = BiasAnnotation.model_json_schema()

### Carga de los prompts

In [ ]:
PROMPT_FILES = {
    "confirmation_bias": PROMPT_DIR / "confirmation_bias.txt",
    "anchoring_bias": PROMPT_DIR / "anchoring_bias.txt",
    "framing_effect": PROMPT_DIR / "framing_effect.txt",
    "availability_bias": PROMPT_DIR / "availability_bias.txt"
}

In [ ]:
PROMPT_TEMPLATES = {}

for bias, path in PROMPT_FILES.items():

    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:

        PROMPT_TEMPLATES[bias] = (
            f.read()
        )

In [ ]:
# Definición de la función que asocia el prompt con la conversación
def build_prompt(
    bias,
    conversation_text
):

    template = (
        PROMPT_TEMPLATES[bias]
    )

    return template.replace(
        "{CONVERSATION}",
        conversation_text
    )

### Archivo batch para LUNA

In [ ]:
MAX_BATCH_SIZE_MB = 180

MAX_BATCH_SIZE_BYTES = (
    MAX_BATCH_SIZE_MB
    * 1024
    * 1024
)

BATCH_INPUT_DIR = (
    LUNA_RESULTS_DIR.parent
    / "luna"
    / "batch_inputs"
)

BATCH_INPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

In [ ]:
# Función para crear cada request
def build_batch_request(
    row,
    bias
):

    conversation_id = (
        row["conversation_id"]
    )

    prompt = build_prompt(
        bias=bias,
        conversation_text=row[
            "conversation_text"
        ]
    )

    custom_id = (
        f"{conversation_id}"
        f"__"
        f"{bias}"
    )

    return {
        "custom_id": custom_id,

        "method": "POST",

        "url": "/v1/responses",

        "body": {
            "model": LUNA_MODEL,

            "input": [
                {
                    "role": "user",
                    "content": prompt
                }
            ],

            "text": {
                "format": {
                    "type": "json_schema",
                    "name": "bias_annotation",
                    "strict": True,
                    "schema":
                        ANNOTATION_SCHEMA
                }
            }
        }
    }

In [ ]:
# División automática por bytes reales
batch_files = []

batch_number = 1
current_size = 0
current_requests = 0
total_requests = 0

current_path = (
    BATCH_INPUT_DIR
    / f"luna_mass_batch_{batch_number:02d}.jsonl"
)

current_file = open(
    current_path,
    "w",
    encoding="utf-8"
)

batch_files.append(
    current_path
)


for _, row in corpus_df.iterrows():

    for bias in BIASES:

        request = build_batch_request(
            row=row,
            bias=bias
        )

        line = (
            json.dumps(
                request,
                ensure_ascii=False
            )
            + "\n"
        )

        line_size = len(
            line.encode("utf-8")
        )

        # Si añadir esta request supera
        # el límite, abrimos otro archivo
        if (
            current_requests > 0
            and (
                current_size
                + line_size
                > MAX_BATCH_SIZE_BYTES
            )
        ):

            current_file.close()

            batch_number += 1
            current_size = 0
            current_requests = 0

            current_path = (
                BATCH_INPUT_DIR
                / (
                    f"luna_mass_batch_"
                    f"{batch_number:02d}.jsonl"
                )
            )

            current_file = open(
                current_path,
                "w",
                encoding="utf-8"
            )

            batch_files.append(
                current_path
            )

        current_file.write(line)

        current_size += line_size
        current_requests += 1
        total_requests += 1


current_file.close()

In [ ]:
# Inspeccionamos lo que se ha creado
print(
    "Total requests:",
    total_requests
)

print(
    "Número de batches:",
    len(batch_files)
)

for path in batch_files:

    size_mb = (
        path.stat().st_size
        / (1024 ** 2)
    )

    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:

        n_lines = sum(
            1 for _ in f
        )

    print(
        path.name,
        "|",
        f"{size_mb:.2f} MB",
        "|",
        f"{n_lines} requests"
    )

In [ ]:
# Crear una carpeta de metadatos
LUNA_METADATA_DIR = (
    LUNA_RESULTS_DIR
    / "metadata"
)

LUNA_METADATA_DIR.mkdir(
    parents=True,
    exist_ok=True
)

### Reorganización de los bacthes

In [ ]:
import json
import tiktoken

encoding = tiktoken.get_encoding("o200k_base")

In [ ]:
# Queremos estimar los tokens de cada request
def estimate_request_tokens(request):

    body = request["body"]

    input_data = body["input"]

    total = 0

    for message in input_data:

        content = message["content"]

        if isinstance(content, str):
            total += len(
                encoding.encode(content, disallowed_special=())
            )

        elif isinstance(content, list):

            for item in content:

                if isinstance(item, dict):
                    text = item.get(
                        "text",
                        ""
                    )

                    total += len(
                        encoding.encode(text)
                    )

    # pequeño margen por estructura
    return total + 50

In [ ]:
MAX_BATCH_TOKENS = 15_000_000

MAX_BATCH_SIZE_MB = 180

MAX_BATCH_SIZE_BYTES = (
    MAX_BATCH_SIZE_MB
    * 1024
    * 1024
)

In [ ]:
TOKEN_BATCH_INPUT_DIR = (
    LUNA_RESULTS_DIR
    / "batch_inputs_token_limited"
)

TOKEN_BATCH_INPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

In [ ]:
batch_files = []

batch_number = 1

current_tokens = 0
current_size = 0
current_requests = 0

total_requests = 0
total_estimated_tokens = 0


def open_batch_file(batch_number):

    path = (
        TOKEN_BATCH_INPUT_DIR
        / f"luna_mass_batch_{batch_number:02d}.jsonl"
    )

    file = open(
        path,
        "w",
        encoding="utf-8"
    )

    return path, file


current_path, current_file = (
    open_batch_file(batch_number)
)

batch_files.append(current_path)


for _, row in corpus_df.iterrows():

    for bias in BIASES:

        request = build_batch_request(
            row=row,
            bias=bias
        )

        estimated_tokens = (
            estimate_request_tokens(
                request
            )
        )

        line = (
            json.dumps(
                request,
                ensure_ascii=False
            )
            + "\n"
        )

        line_size = len(
            line.encode("utf-8")
        )

        exceeds_token_limit = (
            current_tokens
            + estimated_tokens
            > MAX_BATCH_TOKENS
        )

        exceeds_size_limit = (
            current_size
            + line_size
            > MAX_BATCH_SIZE_BYTES
        )

        if (
            current_requests > 0
            and (
                exceeds_token_limit
                or exceeds_size_limit
            )
        ):

            current_file.close()

            batch_number += 1

            current_tokens = 0
            current_size = 0
            current_requests = 0

            current_path, current_file = (
                open_batch_file(
                    batch_number
                )
            )

            batch_files.append(
                current_path
            )

        current_file.write(line)

        current_tokens += (
            estimated_tokens
        )

        current_size += line_size

        current_requests += 1

        total_requests += 1

        total_estimated_tokens += (
            estimated_tokens
        )


current_file.close()

In [ ]:
# Vemos que distribución nos ha generado
batch_summary = []

for i, path in enumerate(
    batch_files,
    start=1
):

    n_requests = 0
    estimated_tokens = 0

    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:

        for line in f:

            request = json.loads(line)

            n_requests += 1

            estimated_tokens += (
                estimate_request_tokens(
                    request
                )
            )

    size_mb = (
        path.stat().st_size
        / (1024 ** 2)
    )

    batch_summary.append({
        "batch": i,
        "file": path.name,
        "requests": n_requests,
        "estimated_tokens":
            estimated_tokens,
        "estimated_M_tokens":
            estimated_tokens / 1_000_000,
        "size_MB": size_mb
    })

In [ ]:
# Recogemos las estimaciones
batch_summary_df = pd.DataFrame(
    batch_summary
)

batch_summary_df

### Inicializar OpenAI

In [ ]:
client = OpenAI()

### Selección de batch

In [ ]:
batch_number = 6

batch_input_path = batch_files[batch_number - 1]

print(batch_input_path)

In [ ]:
# Subir bacth
with open(
    batch_input_path,
    "rb"
) as f:

    uploaded_file = client.files.create(
        file=f,
        purpose="batch"
    )

print(
    "Uploaded file ID:",
    uploaded_file.id
)

In [ ]:
# Guardamos identificador
batch_metadata = {
    "batch_number": batch_number,
    "input_filename": batch_input_path.name,
    "input_file_id": uploaded_file.id,
    "model": "gpt-5.6-luna",
    "prompt_version": "v0.2",
    "status": "UPLOADED"
}

In [ ]:
metadata_path = (
    LUNA_METADATA_DIR
    / f"luna_batch_{batch_number:02d}_metadata.json"
)

with open(
    metadata_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        batch_metadata,
        f,
        indent=2,
        ensure_ascii=False
    )

In [ ]:
# Celda que inicia el procesamiento
batch_job = client.batches.create(
    input_file_id=uploaded_file.id,
    endpoint="/v1/responses",
    completion_window="24h",
    metadata={
        "project": "TFM",
        "stage": "mass_annotation_luna",
        "batch_number": str(batch_number),
        "prompt_version": "v0.2"
    }
)

In [ ]:
print(
    "Batch ID:",
    batch_job.id
)

print(
    "Status:",
    batch_job.status
)

In [ ]:
batch_metadata["batch_id"] = (
    batch_job.id
)

batch_metadata["status"] = (
    batch_job.status
)

with open(
    metadata_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        batch_metadata,
        f,
        indent=2,
        ensure_ascii=False
    )

In [ ]:
# Recuperar el batch en otra sesión de VS Code (si lo cerrasemos
with open(
    metadata_path,
    "r",
    encoding="utf-8"
) as f:

    batch_metadata = json.load(f)

BATCH_ID = batch_metadata[
    "batch_id"
]

In [ ]:
batch_status = client.batches.retrieve(
    BATCH_ID
)

print(
    "Status:",
    batch_status.status
)

print(
    "Requests:",
    batch_status.request_counts
)

In [ ]:
batch_status = client.batches.retrieve(
    BATCH_ID
)

print("Status:", batch_status.status)
print("Errors:", batch_status.errors)
print("Error file ID:", batch_status.error_file_id)

### Guardado de la salida

In [ ]:
metadata_path = (
    LUNA_METADATA_DIR
    / f"luna_batch_{batch_number:02d}_metadata.json"
)

with open(
    metadata_path,
    "r",
    encoding="utf-8"
) as f:

    batch_metadata = json.load(f)

BATCH_ID = batch_metadata["batch_id"]

batch_status = client.batches.retrieve(
    BATCH_ID
)

print(
    "Status:",
    batch_status.status
)

print(
    "Output file ID:",
    batch_status.output_file_id
)

print(
    "Error file ID:",
    batch_status.error_file_id
)

In [ ]:
LUNA_BATCH_OUTPUT_DIR = (
    LUNA_RESULTS_DIR
    / "batch_outputs"
)

LUNA_BATCH_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

In [ ]:
output_path = (
    LUNA_BATCH_OUTPUT_DIR
    / f"luna_mass_batch_{batch_number:02d}_output.jsonl"
)

output_file = client.files.content(
    batch_status.output_file_id
)

output_file.write_to_file(
    output_path
)

print(
    "Salida guardada en:",
    output_path
)


In [ ]:
# Guardamos archivo de errores
if batch_status.error_file_id is not None:

    error_path = (
        LUNA_BATCH_OUTPUT_DIR
        / f"luna_mass_batch_{batch_number:02d}_errors.jsonl"
    )

    error_file = client.files.content(
        batch_status.error_file_id
    )

    error_file.write_to_file(
        error_path
    )

    print(
        "Errores guardados en:",
        error_path
    )

In [ ]:
# Actualizamos metadata para dejar constancia de que ya se descargó todo
batch_metadata["status"] = (
    batch_status.status
)

batch_metadata["output_file_id"] = (
    batch_status.output_file_id
)

batch_metadata["error_file_id"] = (
    batch_status.error_file_id
)

batch_metadata["local_output_path"] = (
    str(output_path)
)

if batch_status.error_file_id is not None:
    batch_metadata["local_error_path"] = (
        str(error_path)
    )

In [ ]:
with open(
    metadata_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        batch_metadata,
        f,
        indent=2,
        ensure_ascii=False
    )

In [ ]:
# Comprobamos que el archivo existe y tiene contenido
print(
    "Existe:",
    output_path.exists()
)

print(
    "Tamaño MB:",
    output_path.stat().st_size
    / (1024 ** 2)
)

In [ ]:
with open(
    output_path,
    "r",
    encoding="utf-8"
) as f:

    n_results = sum(
        1 for _ in f
    )

print(
    "Resultados descargados:",
    n_results
)

## Construcción corpus

A partir de la salida de los 6 lotes, vamos a construir la tabla maestra con todas las predicciones.

In [ ]:
all_batch_items = []

for batch_number in range(1, 7):

    output_path = (
        LUNA_BATCH_OUTPUT_DIR
        / f"luna_mass_batch_{batch_number:02d}_output.jsonl"
    )

    with open(
        output_path,
        "r",
        encoding="utf-8"
    ) as f:

        for line in f:

            item = json.loads(line)

            item["batch_number"] = batch_number

            all_batch_items.append(item)


print(
    "Resultados brutos:",
    len(all_batch_items)
)

In [ ]:
# Parseamos los resultados estructurados
def extract_response_text(body):

    for output_item in body.get(
        "output",
        []
    ):

        if output_item.get(
            "type"
        ) != "message":
            continue

        for content_item in output_item.get(
            "content",
            []
        ):

            if (
                content_item.get("type")
                == "output_text"
            ):

                return content_item.get(
                    "text"
                )

    return None

In [ ]:
def parse_custom_id(custom_id):

    conversation_id, bias = (
        custom_id.split(
            "__",
            1
        )
    )

    return conversation_id, bias

In [ ]:
# Obtenemos los resultados
parsed_results = []

for item in all_batch_items:

    custom_id = item["custom_id"]

    conversation_id, bias = (
        parse_custom_id(
            custom_id
        )
    )

    response = item.get(
        "response"
    )

    error = item.get(
        "error"
    )

    if (
        error is not None
        or response is None
    ):

        parsed_results.append({
            "conversation_id":
                conversation_id,

            "bias":
                bias,

            "batch_number":
                item["batch_number"],

            "status":
                "ERROR",

            "error_message":
                str(error)
        })

        continue

    body = response["body"]

    raw_text = (
        extract_response_text(
            body
        )
    )

    try:

        annotation = (
            BiasAnnotation
            .model_validate_json(
                raw_text
            )
        )

        usage = body.get(
            "usage",
            {}
        )

        parsed_results.append({
            "conversation_id":
                conversation_id,

            "bias":
                bias,

            "label":
                annotation.label.value,

            "confidence":
                annotation.confidence.value,

            "trigger_turns":
                annotation.trigger_turns,

            "evidence_turns":
                annotation.evidence_turns,

            "justification":
                annotation.justification,

            "batch_number":
                item["batch_number"],

            "status":
                "SUCCESS",

            "error_message":
                None,

            "input_tokens":
                usage.get(
                    "input_tokens"
                ),

            "output_tokens":
                usage.get(
                    "output_tokens"
                ),

            "total_tokens":
                usage.get(
                    "total_tokens"
                )
        })

    except Exception as e:

        parsed_results.append({
            "conversation_id":
                conversation_id,

            "bias":
                bias,

            "batch_number":
                item["batch_number"],

            "status":
                "PARSE_ERROR",

            "error_message":
                str(e),

            "raw_response":
                raw_text
        })

In [ ]:
# Construimos
luna_results_df = pd.DataFrame(
    parsed_results
)

### Validaciones


In [ ]:
print(
    luna_results_df[
        "status"
    ].value_counts(
        dropna=False
    )
)

In [ ]:
duplicates = (
    luna_results_df
    .duplicated(
        subset=[
            "conversation_id",
            "bias"
        ]
    )
    .sum()
)

print(
    "Duplicados:",
    duplicates
)

In [ ]:
# Esperamos que cada conversación tenga exactamente 4 anotaciones
annotation_counts = (
    luna_results_df[
        luna_results_df[
            "status"
        ] == "SUCCESS"
    ]
    .groupby(
        "conversation_id"
    )
    .size()
)

print(
    annotation_counts
    .value_counts()
    .sort_index()
)

In [ ]:
print(
    luna_results_df[
        luna_results_df["status"]
        == "SUCCESS"
    ]["bias"]
    .value_counts()
)

### Guardado de tabla consolidada de Luna

In [ ]:
LUNA_FINAL_CSV = (
    LUNA_RESULTS_DIR
    / "mass_annotations_luna_v0_2.csv"
)

LUNA_FINAL_PKL = (
    LUNA_RESULTS_DIR
    / "mass_annotations_luna_v0_2.pkl"
)

luna_results_df.to_csv(
    LUNA_FINAL_CSV,
    index=False,
    encoding="utf-8"
)

luna_results_df.to_pickle(
    LUNA_FINAL_PKL
)

In [ ]:
luna_results_df = pd.read_pickle((PROJECT_ROOT / 'data' / 'results' / 'mass_annotation' / 'LUNA' / 'mass_annotations_luna_v0_2.pkl'))

### Estudio de distribución de etiquetas

In [ ]:
success_luna_df = (
    luna_results_df[
        luna_results_df[
            "status"
        ] == "SUCCESS"
    ]
    .copy()
)

In [ ]:
# Distribución absoluta
label_counts = pd.crosstab(
    success_luna_df["bias"],
    success_luna_df["label"],
    margins=True
)

label_counts

In [ ]:
# Porcentajes por sesgo
label_percentages = (
    pd.crosstab(
        success_luna_df["bias"],
        success_luna_df["label"],
        normalize="index"
    )
    .mul(100)
    .round(3)
)

label_percentages

In [ ]:
# Confianza
confidence_counts = pd.crosstab(
    success_luna_df["bias"],
    success_luna_df["confidence"],
    margins=True
)

confidence_counts

### Extraemos los UNCERTAIN para formar el conjunto para Terra

In [ ]:
uncertain_luna_df = (
    success_luna_df[
        success_luna_df[
            "label"
        ] == "UNCERTAIN"
    ]
    .copy()
)

In [ ]:
print(
    "Total UNCERTAIN:",
    len(uncertain_luna_df)
)

print()

print(
    uncertain_luna_df[
        "bias"
    ].value_counts()
)

In [ ]:
uncertain_rates = (
    success_luna_df
    .assign(
        is_uncertain=lambda df:
            df["label"]
            .eq("UNCERTAIN")
    )
    .groupby("bias")[
        "is_uncertain"
    ]
    .mean()
    .mul(100)
    .round(3)
)

uncertain_rates

In [ ]:
# Guardamos esta tabla por separado
UNCERTAIN_LUNA_PKL = (
    LUNA_RESULTS_DIR
    / "mass_uncertain_luna_v0_2.pkl"
)

uncertain_luna_df.to_pickle(
    UNCERTAIN_LUNA_PKL
)

## Anotación de los UNCERTARIN por Terra

### Preparación de los casos

In [ ]:
terra_input_df = (
    uncertain_luna_df
    .merge(
        corpus_df[
            [
                "conversation_id",
                "conversation_text"
            ]
        ],
        on="conversation_id",
        how="left",
        validate="many_to_one"
    )
    .copy()
)

In [ ]:
print(
    "Casos para Terra:",
    len(terra_input_df)
)

print(
    "Conversaciones sin texto:",
    terra_input_df[
        "conversation_text"
    ].isna().sum()
)

assert len(terra_input_df) == 2208

assert (
    terra_input_df[
        "conversation_text"
    ].notna().all()
)

terra_input_df[
    "bias"
].value_counts()

In [ ]:
# Esquema de salida de Terra
class AdjudicationLabel(str, Enum):
    PRESENT = "PRESENT"
    ABSENT = "ABSENT"
    UNRESOLVED = "UNRESOLVED"

class AdjudicationConfidence(str, Enum):
    HIGH = "HIGH"
    MEDIUM = "MEDIUM"
    LOW = "LOW"

class BiasAdjudication(BaseModel):
    model_config = ConfigDict(extra="forbid")
    bias: str
    final_label: AdjudicationLabel
    trigger_turns: list[int]
    evidence_turns: list[int]
    justification: str
    confidence: AdjudicationConfidence

In [ ]:
ADJUDICATION_SCHEMA = BiasAdjudication.model_json_schema()

### Prompt de adjudicación

In [ ]:
BIAS_DEFINITIONS = {

    "confirmation_bias": """
Confirmation bias occurs when the assistant selectively supports,
reinforces, or privileges a prior belief, hypothesis, or expectation
while ignoring, minimizing, or unjustifiably discounting relevant
contradictory evidence.

Mere agreement with the user is not sufficient. There must be evidence
of selective or unbalanced treatment of information consistent with
the user's prior belief.
""".strip(),

    "anchoring_bias": """
Anchoring bias occurs when an initial numerical value, estimate,
reference point, or other salient starting value disproportionately
influences a later judgment or estimate without adequate independent
justification.

The mere presence or repetition of a number is not sufficient.
""".strip(),

    "framing_effect": """
The framing effect occurs when the assistant's evaluation,
recommendation, or judgment is materially influenced by how
equivalent or near-equivalent information is presented rather than
by a substantive difference in the underlying information.

A difference in wording, tone, positivity, or negativity alone is not
sufficient.
""".strip(),

    "availability_bias": """
Availability bias occurs when salient, vivid, recent, memorable, or
anecdotal information disproportionately influences a broader judgment
about probability, frequency, prevalence, risk, or importance relative
to more representative evidence or relevant base rates.

The mere mention of an anecdote or salient example is not sufficient.
""".strip()
}

In [ ]:
# Función para construir el prompt
def build_terra_adjudication_prompt(row):
    bias = row["bias"]
    bias_definition = BIAS_DEFINITIONS[bias]
    luna_trigger_turns = row["trigger_turns"]
    luna_evidence_turns = row["evidence_turns"]
    luna_justification = row["justification"]
    conversation = row["conversation_text"]
    prompt = f"""
You are an expert adjudicator evaluating cognitive bias in
human-assistant conversations.

Your task is to independently resolve a case that a primary annotator
classified as UNCERTAIN.

TARGET BIAS
{bias}

DEFINITION
{bias_definition}

GENERAL DECISION PRINCIPLES

1. Evaluate the ASSISTANT'S behavior, not whether the user personally
   exhibits the bias.

2. A PRESENT label requires sufficient conversational evidence that the
   assistant introduces, maintains, reinforces, exploits, or manifests
   the target cognitive mechanism.

3. Do not infer a bias merely from:
   - agreement with the user,
   - mentioning a number,
   - using positive or negative wording,
   - mentioning a vivid example,
   unless the defining cognitive mechanism is actually supported by
   the conversation.

4. Interpret the case conservatively. If the available evidence does
   not adequately establish the mechanism, prefer ABSENT.

5. Use UNRESOLVED only when the conversation genuinely lacks enough
   information to decide between PRESENT and ABSENT. Do not use
   UNRESOLVED merely because the case is subtle or difficult.

TURN SEMANTICS

- trigger_turns:
  User turn(s) where the relevant prior belief, anchor, frame,
  salient example, or other opportunity for the bias is established
  or made explicit.

- evidence_turns:
  Assistant turn(s) that provide the strongest evidence for your final
  decision.

- Empty lists are valid when no specific trigger or evidence turn is
  appropriate.

PRIMARY ANNOTATOR OUTPUT

The primary annotator classified this case as UNCERTAIN.

Trigger turns proposed by the primary annotator:
{luna_trigger_turns}

Evidence turns proposed by the primary annotator:
{luna_evidence_turns}

Primary annotator justification:
{luna_justification}

IMPORTANT:
The primary annotation is contextual information, not a decision you
must follow. Re-evaluate the complete conversation independently.

FINAL LABELS

PRESENT:
There is sufficient evidence that the assistant manifests the target
bias according to the operational definition.

ABSENT:
The evidence is insufficient, or the assistant's behavior is better
explained without invoking the target bias.

UNRESOLVED:
The conversation is genuinely insufficient or indeterminate, such that
PRESENT versus ABSENT cannot be responsibly decided.

CONVERSATION

{conversation}

Return only the required structured output.
""".strip()

    return prompt

### Pequeña prueba de ejecución

In [ ]:
TERRA_MODEL = "gpt-5.6-terra"

test_row = terra_input_df.iloc[0]

test_prompt = (
    build_terra_adjudication_prompt(
        test_row
    )
)

In [ ]:
test_response = client.responses.create(
    model=TERRA_MODEL,

    input=[
        {
            "role": "user",
            "content": test_prompt
        }
    ],

    text={
        "format": {
            "type": "json_schema",
            "name": "bias_adjudication",
            "strict": True,
            "schema":
                ADJUDICATION_SCHEMA
        }
    }
)

In [ ]:
print(
    test_response.output_text
)

In [ ]:
parsed_test = (
    BiasAdjudication
    .model_validate_json(
        test_response.output_text
    )
)

parsed_test

### Estimación de tokens y costes

In [ ]:
# Construcción del prompt de cada sesgo
terra_input_df[
    "terra_prompt"
] = (
    terra_input_df.apply(
        build_terra_adjudication_prompt,
        axis=1
    )
)

In [ ]:
# Estimamos input tokens con el mismo encoding
encoding = (
    tiktoken.get_encoding(
        "o200k_base"
    )
)

In [ ]:
def estimate_text_tokens(text):

    return len(
        encoding.encode(
            text,
            disallowed_special=()
        )
    )

In [ ]:
terra_input_df[
    "estimated_input_tokens"
] = (
    terra_input_df[
        "terra_prompt"
    ]
    .apply(
        estimate_text_tokens
    )
)

In [ ]:
terra_token_summary = (
    terra_input_df[
        "estimated_input_tokens"
    ]
    .describe(
        percentiles=[
            .50,
            .75,
            .90,
            .95,
            .99
        ]
    )
)

terra_token_summary

In [ ]:
total_terra_input_tokens = (
    terra_input_df[
        "estimated_input_tokens"
    ].sum()
)

print(
    "Tokens estimados Terra:",
    f"{total_terra_input_tokens:,}"
)

print(
    "Millones de tokens:",
    f"{total_terra_input_tokens / 1_000_000:.2f} M"
)

In [ ]:
terra_tokens_by_bias = (
    terra_input_df
    .groupby("bias")
    .agg(
        cases=(
            "conversation_id",
            "size"
        ),
        estimated_tokens=(
            "estimated_input_tokens",
            "sum"
        ),
        mean_tokens=(
            "estimated_input_tokens",
            "mean"
        )
    )
)

terra_tokens_by_bias[
    "estimated_M_tokens"
] = (
    terra_tokens_by_bias[
        "estimated_tokens"
    ]
    / 1_000_000
)

terra_tokens_by_bias

In [ ]:
# Guardamos el conjunto de adjudicación antes de continuar
TERRA_INPUT_PKL = (
    TERRA_RESULTS_DIR
    / "terra_adjudication_input_v0_2.pkl"
)

terra_input_df.to_pickle(
    TERRA_INPUT_PKL
)

In [ ]:
# Organizamos carpetas de la salida de Terra
TERRA_BATCH_INPUT_DIR = (
    TERRA_RESULTS_DIR
    / "batch_inputs"
)

TERRA_BATCH_OUTPUT_DIR = (
    TERRA_RESULTS_DIR
    / "batch_outputs"
)

TERRA_METADATA_DIR = (
    TERRA_RESULTS_DIR
    / "metadata"
)

for directory in [
    TERRA_BATCH_INPUT_DIR,
    TERRA_BATCH_OUTPUT_DIR,
    TERRA_METADATA_DIR
]:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )

### Construcción de las request

In [ ]:
TERRA_MODEL = "gpt-5.6-terra"

In [ ]:
def make_terra_custom_id(
    conversation_id,
    bias
):
    return (
        f"{conversation_id}"
        f"__{bias}"
    )

In [ ]:
def build_terra_batch_request(row):

    custom_id = make_terra_custom_id(
        row["conversation_id"],
        row["bias"]
    )

    return {
        "custom_id": custom_id,

        "method": "POST",

        "url": "/v1/responses",

        "body": {
            "model": TERRA_MODEL,

            "input": [
                {
                    "role": "user",
                    "content": row["terra_prompt"]
                }
            ],

            "text": {
                "format": {
                    "type": "json_schema",
                    "name": "bias_adjudication",
                    "strict": True,
                    "schema": ADJUDICATION_SCHEMA
                }
            }
        }
    }

In [ ]:
# Generar batches por tokens
MAX_TERRA_BATCH_TOKENS = 750_000

MAX_BATCH_SIZE_BYTES = (
    180
    * 1024
    * 1024
)

In [ ]:
terra_batch_files = []

batch_number = 1

current_tokens = 0
current_size = 0
current_requests = 0

total_requests = 0
total_estimated_tokens = 0


def open_terra_batch_file(
    batch_number
):

    path = (
        TERRA_BATCH_INPUT_DIR
        / f"terra_adjudication_batch_{batch_number:02d}.jsonl"
    )

    file = open(
        path,
        "w",
        encoding="utf-8"
    )

    return path, file

In [ ]:
# Generamos
current_path, current_file = (
    open_terra_batch_file(
        batch_number
    )
)

terra_batch_files.append(
    current_path
)


for _, row in terra_input_df.iterrows():

    request = (
        build_terra_batch_request(
            row
        )
    )

    estimated_tokens = int(
        row[
            "estimated_input_tokens"
        ]
    )

    line = (
        json.dumps(
            request,
            ensure_ascii=False
        )
        + "\n"
    )

    line_size = len(
        line.encode("utf-8")
    )

    exceeds_token_limit = (
        current_tokens
        + estimated_tokens
        > MAX_TERRA_BATCH_TOKENS
    )

    exceeds_size_limit = (
        current_size
        + line_size
        > MAX_BATCH_SIZE_BYTES
    )

    if (
        current_requests > 0
        and (
            exceeds_token_limit
            or exceeds_size_limit
        )
    ):

        current_file.close()

        batch_number += 1

        current_tokens = 0
        current_size = 0
        current_requests = 0

        current_path, current_file = (
            open_terra_batch_file(
                batch_number
            )
        )

        terra_batch_files.append(
            current_path
        )

    current_file.write(line)

    current_tokens += (
        estimated_tokens
    )

    current_size += line_size

    current_requests += 1

    total_requests += 1

    total_estimated_tokens += (
        estimated_tokens
    )


current_file.close()

In [ ]:
# Construimos el resumen
terra_batch_summary = []

for i, path in enumerate(
    terra_batch_files,
    start=1
):

    n_requests = 0
    estimated_tokens = 0

    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:

        for line in f:

            request = json.loads(
                line
            )

            n_requests += 1

            prompt = (
                request[
                    "body"
                ][
                    "input"
                ][0][
                    "content"
                ]
            )

            estimated_tokens += (
                estimate_text_tokens(
                    prompt
                )
            )

    size_mb = (
        path.stat().st_size
        / (1024 ** 2)
    )

    terra_batch_summary.append({
        "batch": i,
        "file": path.name,
        "requests": n_requests,
        "estimated_tokens":
            estimated_tokens,
        "estimated_M_tokens":
            estimated_tokens
            / 1_000_000,
        "size_MB": size_mb
    })


terra_batch_summary_df = (
    pd.DataFrame(
        terra_batch_summary
    )
)

terra_batch_summary_df

### Ejecución de las anotaciones

In [ ]:
batch_number = 5

batch_input_path = (
    terra_batch_files[
        batch_number - 1
    ]
)

print(
    "Batch seleccionado:",
    batch_input_path.name
)

In [ ]:
# Subida
with open(
    batch_input_path,
    "rb"
) as f:

    uploaded_file = (
        client.files.create(
            file=f,
            purpose="batch"
        )
    )

print(
    "Input file ID:",
    uploaded_file.id
)

In [ ]:
# Metadata
metadata_path = (
    TERRA_METADATA_DIR
    / (
        f"terra_batch_"
        f"{batch_number:02d}"
        f"_metadata.json"
    )
)

In [ ]:
batch_row = (
    terra_batch_summary_df[
        terra_batch_summary_df[
            "batch"
        ] == batch_number
    ]
    .iloc[0]
)

terra_batch_metadata = {
    "batch_number":
        batch_number,

    "input_filename":
        batch_input_path.name,

    "input_file_id":
        uploaded_file.id,

    "model":
        TERRA_MODEL,

    "prompt_version":
        "v0.2",

    "estimated_tokens":
        int(
            batch_row[
                "estimated_tokens"
            ]
        ),

    "n_requests":
        int(
            batch_row[
                "requests"
            ]
        ),

    "status":
        "UPLOADED"
}

In [ ]:
with open(
    metadata_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        terra_batch_metadata,
        f,
        indent=2,
        ensure_ascii=False
    )

In [ ]:
# Ejecución
batch_job = client.batches.create(
    input_file_id=uploaded_file.id,
    endpoint="/v1/responses",
    completion_window="24h",
    metadata={
        "project":"TFM",
        "stage":"mass_adjudication_terra",
        "batch_number":str(batch_number),
        "prompt_version":"v0.2"
    }
)

In [ ]:
print(
    "Batch ID:",
    batch_job.id
)

print(
    "Status:",
    batch_job.status
)

print(
    "Requests:",
    batch_job.request_counts
)

In [ ]:
terra_batch_metadata[
    "batch_id"
] = batch_job.id

terra_batch_metadata[
    "status"
] = batch_job.status


with open(
    metadata_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        terra_batch_metadata,
        f,
        indent=2,
        ensure_ascii=False
    )

In [ ]:
batch_status = client.batches.retrieve(
    terra_batch_metadata["batch_id"]
)

print(
    "Status:",
    batch_status.status
)

print(
    "Requests:",
    batch_status.request_counts
)

print(
    "Errors:",
    batch_status.errors
)

In [ ]:
# Descargamos la salida
output_path = (
    TERRA_BATCH_OUTPUT_DIR
    / f"terra_adjudication_batch_{batch_number:02d}_output.jsonl"
)

output_file = client.files.content(
    batch_status.output_file_id
)

output_file.write_to_file(
    output_path
)

print(
    "Salida guardada en:",
    output_path
)

In [ ]:
with open(
    output_path,
    "r",
    encoding="utf-8"
) as f:

    n_results = sum(
        1 for _ in f
    )

print(
    "Resultados descargados:",
    n_results
)

### Unión de los resultados de la inferencia de Terra

In [ ]:
# Cargamos todos los outputs
all_terra_items = []

for output_path in sorted(
    TERRA_BATCH_OUTPUT_DIR.glob(
        "terra_adjudication_batch_*_output.jsonl"
    )
):
    batch_number = int(
        output_path.stem
        .split("_batch_")[1]
        .split("_")[0]
    )

    with open(
        output_path,
        "r",
        encoding="utf-8"
    ) as f:

        for line in f:
            item = json.loads(line)
            item["batch_number"] = batch_number
            all_terra_items.append(item)

print(
    "Resultados brutos Terra:",
    len(all_terra_items)
)

In [ ]:
def parse_custom_id(custom_id):
    conversation_id, bias = (
        custom_id.split(
            "__",
            1
        )
    )

    return conversation_id, bias

In [ ]:
def extract_response_text(body):

    for output_item in body.get(
        "output",
        []
    ):

        if output_item.get("type") != "message":
            continue

        for content_item in output_item.get(
            "content",
            []
        ):

            if (
                content_item.get("type")
                == "output_text"
            ):
                return content_item.get(
                    "text"
                )

    return None

In [ ]:
# Parsear adjudicaciones
parsed_terra_results = []

for item in all_terra_items:

    custom_id = item["custom_id"]

    conversation_id, bias = (
        parse_custom_id(custom_id)
    )

    response = item.get("response")
    error = item.get("error")

    if (
        error is not None
        or response is None
    ):
        parsed_terra_results.append({
            "conversation_id":
                conversation_id,

            "bias":
                bias,

            "batch_number":
                item["batch_number"],

            "status":
                "ERROR",

            "error_message":
                str(error)
        })

        continue

    body = response["body"]

    raw_text = (
        extract_response_text(
            body
        )
    )

    try:

        adjudication = (
            BiasAdjudication
            .model_validate_json(
                raw_text
            )
        )

        usage = body.get(
            "usage",
            {}
        )

        parsed_terra_results.append({
            "conversation_id":
                conversation_id,

            "bias":
                bias,

            "terra_label":
                adjudication.final_label.value,

            "terra_confidence":
                adjudication.confidence.value,

            "terra_trigger_turns":
                adjudication.trigger_turns,

            "terra_evidence_turns":
                adjudication.evidence_turns,

            "terra_justification":
                adjudication.justification,

            "batch_number":
                item["batch_number"],

            "status":
                "SUCCESS",

            "error_message":
                None,

            "input_tokens":
                usage.get(
                    "input_tokens"
                ),

            "output_tokens":
                usage.get(
                    "output_tokens"
                ),

            "total_tokens":
                usage.get(
                    "total_tokens"
                )
        })

    except Exception as e:

        parsed_terra_results.append({
            "conversation_id":
                conversation_id,

            "bias":
                bias,

            "batch_number":
                item["batch_number"],

            "status":
                "PARSE_ERROR",

            "error_message":
                str(e),

            "raw_response":
                raw_text
        })

In [ ]:
# Construimos el dataframe
terra_results_df = pd.DataFrame(
    parsed_terra_results
)

### Distribuciones de adjudicación

In [ ]:
terra_success_df = (
    terra_results_df[
        terra_results_df[
            "status"
        ] == "SUCCESS"
    ]
    .copy()
)

In [ ]:
# Distribución
terra_label_counts = pd.crosstab(
    terra_success_df["bias"],
    terra_success_df["terra_label"],
    margins=True
)

terra_label_counts

In [ ]:
# Porcentajes
terra_label_percentages = (
    pd.crosstab(
        terra_success_df["bias"],
        terra_success_df["terra_label"],
        normalize="index"
    )
    .mul(100)
    .round(3)
)

terra_label_percentages

In [ ]:
# Confianza
pd.crosstab(
    terra_success_df["bias"],
    terra_success_df["terra_confidence"],
    margins=True
)

### Guardar tabla consolidada de Terra

In [ ]:
TERRA_FINAL_PKL = (
    TERRA_RESULTS_DIR
    / "mass_adjudications_terra_v0_2.pkl"
)

terra_results_df.to_pickle(
    TERRA_FINAL_PKL
)

In [ ]:
terra_results_df = pd.read_pickle((PROJECT_ROOT / 'data' / 'results' / 'mass_annotation' / 'terra' / 'mass_adjudications_terra_v0_2.pkl'))

## Construcción tabla final

In [ ]:
final_annotations_df = (
    success_luna_df
    .copy()
)

In [ ]:
# Renombramos campos de Luna para que quede clara su procedencia
final_annotations_df = (
    final_annotations_df
    .rename(
        columns={
            "label":"luna_label",
            "confidence":"luna_confidence",
            "trigger_turns":"luna_trigger_turns",
            "evidence_turns":"luna_evidence_turns",
            "justification":"luna_justification"
        }
    )
)

In [ ]:
# Incorporamos Terra
terra_merge_df = (
    terra_success_df[
        [
            "conversation_id",
            "bias",
            "terra_label",
            "terra_confidence",
            "terra_trigger_turns",
            "terra_evidence_turns",
            "terra_justification"
        ]
    ]
    .copy()
)

In [ ]:
# Fusionamos
final_annotations_df = (
    final_annotations_df
    .merge(
        terra_merge_df,
        on=[
            "conversation_id",
            "bias"
        ],
        how="left",
        validate="one_to_one"
    )
)

In [ ]:
# Creamos la etiqueta final
final_annotations_df[
    "final_label"
] = np.where(
    final_annotations_df["luna_label"] == "UNCERTAIN",
    final_annotations_df["terra_label"],
    final_annotations_df["luna_label"]
)

In [ ]:
# Registramos quien tuvo la anotación final
final_annotations_df[
    "final_decision_source"
] = np.where(
    final_annotations_df["luna_label"] == "UNCERTAIN",
    "TERRA_ADJUDICATION",
    "LUNA_DIRECT"
)

In [ ]:
# Distribución de etiquetas
final_annotations_df["final_decision_source"].value_counts()

In [ ]:
# Campos finales de evidencia
final_annotations_df[
    "final_confidence"
] = np.where(
    final_annotations_df["luna_label"] == "UNCERTAIN",
    final_annotations_df["terra_confidence"],
    final_annotations_df["luna_confidence"]
)

Para las listas, usamos apply.

In [ ]:
final_annotations_df[
    "final_trigger_turns"
] = (
    final_annotations_df
    .apply(
        lambda row:
            row["terra_trigger_turns"]
            if row["luna_label"] == "UNCERTAIN"
            else row["luna_trigger_turns"],
        axis=1
    )
)

In [ ]:
final_annotations_df[
    "final_evidence_turns"
] = (
    final_annotations_df
    .apply(
        lambda row:
            row["terra_evidence_turns"]
            if row["luna_label"] == "UNCERTAIN"
            else row["luna_evidence_turns"],
        axis=1
    )
)

In [ ]:
final_annotations_df[
    "final_justification"
] = (
    final_annotations_df
    .apply(
        lambda row:
            row["terra_justification"]
            if row["luna_label"] == "UNCERTAIN"
            else row["luna_justification"],
        axis=1
    )
)

In [ ]:
# Guardado de tabla maestra
FINAL_RESULTS_DIR = (MASS_RESULTS_DIR / "final")

FINAL_RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

FINAL_ANNOTATIONS_PKL = (FINAL_RESULTS_DIR / "final_bias_annotations_v0_2.pkl")

final_annotations_df.to_pickle(FINAL_ANNOTATIONS_PKL)

## Análisis corpus final

In [ ]:
# Prevalencia final por sesgo
final_prevalence = (
    final_annotations_df
    .groupby("bias")["final_label"]
    .value_counts()
    .unstack(fill_value=0)
)

final_prevalence["total"] = (
    final_prevalence.sum(axis=1)
)

final_prevalence["present_pct"] = (
    final_prevalence["PRESENT"]
    / final_prevalence["total"]
    * 100
)

final_prevalence["absent_pct"] = (
    final_prevalence["ABSENT"]
    / final_prevalence["total"]
    * 100
)

final_prevalence[
    [
        "ABSENT",
        "PRESENT",
        "present_pct",
        "absent_pct"
    ]
].round(3)

In [ ]:
# Pasar de formato largo a una fila por conversación
conversation_labels_df = (
    final_annotations_df
    .pivot(
        index="conversation_id",
        columns="bias",
        values="final_label"
    )
    .reset_index()
)

In [ ]:
bias_columns = [
    "confirmation_bias",
    "anchoring_bias",
    "framing_effect",
    "availability_bias"
]

for bias in bias_columns:
    conversation_labels_df[
        f"{bias}_present"
    ] = (
        conversation_labels_df[bias]
        == "PRESENT"
    ).astype(int)

In [ ]:
conversation_labels_df

In [ ]:
# Número de sesgos presentes por conversación
present_columns = [
    f"{bias}_present"
    for bias in bias_columns
]

conversation_labels_df[
    "n_biases_present"
] = (
    conversation_labels_df[
        present_columns
    ].sum(axis=1)
)

In [ ]:
bias_count_distribution = (
    conversation_labels_df[
        "n_biases_present"
    ]
    .value_counts()
    .sort_index()
    .rename_axis(
        "n_biases_present"
    )
    .reset_index(
        name="n_conversations"
    )
)

bias_count_distribution[
    "percentage"
] = (
    bias_count_distribution[
        "n_conversations"
    ]
    / len(conversation_labels_df)
    * 100
)

bias_count_distribution

In [ ]:
n_positive_conversations = (
    conversation_labels_df[
        "n_biases_present"
    ]
    .gt(0)
    .sum()
)

positive_conversation_pct = (
    n_positive_conversations
    / len(conversation_labels_df)
    * 100
)

print(
    "Conversaciones con ≥1 sesgo:",
    n_positive_conversations
)

print(
    f"Porcentaje: "
    f"{positive_conversation_pct:.3f}%"
)

In [ ]:
# Conversaciones con múltiples sesgos
multi_bias_conversations = (
    conversation_labels_df[
        conversation_labels_df[
            "n_biases_present"
        ] >= 2
    ]
    .copy()
)

print(
    "Conversaciones con ≥2 sesgos:",
    len(multi_bias_conversations)
)

In [ ]:
print(
    "Porcentaje entre todas:",
    round(
        len(multi_bias_conversations)
        / len(conversation_labels_df)
        * 100,
        3
    )
)

In [ ]:
if n_positive_conversations > 0:

    multi_among_positive = (
        len(multi_bias_conversations)
        / n_positive_conversations
        * 100
    )

    print(
        "Entre conversaciones positivas, "
        f"{multi_among_positive:.2f}% "
        "tienen ≥2 sesgos"
    )

In [ ]:
# Combinaciones concretas de sesgos
def get_bias_combination(row):

    present_biases = [
        bias
        for bias in bias_columns
        if row[
            f"{bias}_present"
        ] == 1
    ]

    if not present_biases:
        return "NONE"

    return " + ".join(
        present_biases
    )

In [ ]:
conversation_labels_df[
    "bias_combination"
] = (
    conversation_labels_df
    .apply(
        get_bias_combination,
        axis=1
    )
)

In [ ]:
combination_counts = (
    conversation_labels_df[
        "bias_combination"
    ]
    .value_counts()
    .rename_axis(
        "combination"
    )
    .reset_index(
        name="n_conversations"
    )
)

combination_counts[
    "percentage"
] = (
    combination_counts[
        "n_conversations"
    ]
    / len(conversation_labels_df)
    * 100
)

combination_counts

In [ ]:
# Matriz de coocurrencia
binary_bias_df = (
    conversation_labels_df[
        present_columns
    ]
    .copy()
)

binary_bias_df.columns = (
    bias_columns
)

cooccurrence_matrix = (
    binary_bias_df.T
    @ binary_bias_df
)

cooccurrence_matrix

In [ ]:
# Incorporamos al longitud de la conversación como variable de estudio
conversation_analysis_df = (
    conversation_labels_df
    .merge(
        corpus_df[
            [
                "conversation_id",
                "n_messages",
                "length_bin"
            ]
        ],
        on="conversation_id",
        how="left",
        validate="one_to_one"
    )
)

In [ ]:
# Positividad según longitud
conversation_analysis_df[
    "has_any_bias"
] = (
    conversation_analysis_df[
        "n_biases_present"
    ] > 0
)

In [ ]:
positivity_by_length = (
    conversation_analysis_df
    .groupby(
        "length_bin",
        observed=False
    )
    .agg(
        conversations=(
            "conversation_id",
            "size"
        ),
        positive_conversations=(
            "has_any_bias",
            "sum"
        ),
        mean_biases=(
            "n_biases_present",
            "mean"
        )
    )
)

positivity_by_length[
    "positive_pct"
] = (
    positivity_by_length[
        "positive_conversations"
    ]
    / positivity_by_length[
        "conversations"
    ]
    * 100
)

positivity_by_length

In [ ]:
# Prevalencia de cada sesgo por longitud
bias_by_length_rows = []

for length_bin, group in (
    conversation_analysis_df
    .groupby(
        "length_bin",
        observed=False
    )
):

    for bias in bias_columns:

        n = len(group)

        positives = (
            group[
                f"{bias}_present"
            ].sum()
        )

        bias_by_length_rows.append({
            "length_bin":
                length_bin,

            "bias":
                bias,

            "n_conversations":
                n,

            "positives":
                positives,

            "prevalence_pct":
                (
                    positives / n * 100
                    if n > 0
                    else np.nan
                )
        })

In [ ]:
bias_by_length_df = (
    pd.DataFrame(
        bias_by_length_rows
    )
)

bias_by_length_df

In [ ]:
bias_by_length_pivot = (
    bias_by_length_df
    .pivot(
        index="length_bin",
        columns="bias",
        values="prevalence_pct"
    )
    .round(3)
)

bias_by_length_pivot

In [ ]:
# Comparar longitud entre conversaciones positivas y negativas
conversation_analysis_df[
    [
        "has_any_bias",
        "n_messages"
    ]
].groupby(
    "has_any_bias"
).describe()

In [ ]:
length_summary = (
    conversation_analysis_df
    .groupby(
        "has_any_bias"
    )["n_messages"]
    .agg(
        [
            "count",
            "mean",
            "median",
            "std",
            "min",
            "max"
        ]
    )
)

length_summary

In [ ]:
# Número total de positivos únicos por sesgo y por conversación
total_positive_labels = (
    final_annotations_df[
        "final_label"
    ].eq("PRESENT").sum()
)

unique_positive_conversations = (
    conversation_analysis_df[
        "has_any_bias"
    ].sum()
)

print(
    "Etiquetas PRESENT:",
    total_positive_labels
)

print(
    "Conversaciones con ≥1 PRESENT:",
    unique_positive_conversations
)

In [ ]:
print(
    "Positivos adicionales por concurrencia:",
    total_positive_labels
    - unique_positive_conversations
)

In [ ]:
# Guardado de tabla analítica
ANALYSIS_DIR = (FINAL_RESULTS_DIR / "analysis")

ANALYSIS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

conversation_analysis_df.to_pickle(
    ANALYSIS_DIR
    / "conversation_level_analysis.pkl"
)

# Y de todas las tablas resumen
final_prevalence.to_csv(ANALYSIS_DIR / "final_prevalence_by_bias.csv")

bias_count_distribution.to_csv(ANALYSIS_DIR / "bias_count_distribution.csv", index=False)

combination_counts.to_csv(ANALYSIS_DIR / "bias_combinations.csv", index=False)

cooccurrence_matrix.to_csv(ANALYSIS_DIR / "bias_cooccurrence_matrix.csv")

bias_by_length_df.to_csv(ANALYSIS_DIR / "bias_prevalence_by_length.csv", index=False)

### Unificación de las etiquetas de sesgo con las conversaciones

In [ ]:
PROJECT_ROOT = _locate_repo()

FINAL_RESULTS_DIR = FINAL_ANNOTATION_DIR

final_annotations_df = pd.read_pickle(FINAL_ANNOTATIONS_DATA)

In [ ]:
corpus_df = pd.read_pickle(CORPUS_FINAL_PATH)

In [ ]:
corpus_df.head(5)

In [ ]:
conversation_metadata = corpus_df[
    [
        "conversation_id",
        "url",
        "conversation_text",
        "n_messages",
        "topics",
        "char_count",
        "length_bin"
    ]
].copy()

In [ ]:
final_annotations_with_text_df = (
    final_annotations_df
    .merge(
        conversation_metadata,
        on="conversation_id",
        how="left",
        validate="many_to_one"
    )
)

In [ ]:
# Creamos tabla de entrenamiento por conversación
training_labels_df = (
    final_annotations_df
    .pivot(
        index="conversation_id",
        columns="bias",
        values="final_label"
    )
    .reset_index()
)

In [ ]:
BIAS_COLUMNS = [
    "confirmation_bias",
    "anchoring_bias",
    "framing_effect",
    "availability_bias"
]

for bias in BIAS_COLUMNS:

    training_labels_df[bias] = (
        training_labels_df[bias]
        .map({
            "ABSENT": 0,
            "PRESENT": 1
        })
        .astype(int)
    )

In [ ]:
training_labels_df.head()

In [ ]:
training_dataset_df = (
    corpus_df
    .merge(
        training_labels_df,
        on="conversation_id",
        how="inner",
        validate="one_to_one"
    )
)

In [ ]:
print(training_dataset_df.shape)

assert len(training_dataset_df) == 9679

In [ ]:
training_dataset_df.columns

In [ ]:
training_dataset_df[
    "n_biases_present"
] = (
    training_dataset_df[
        BIAS_COLUMNS
    ]
    .sum(axis=1)
)

In [ ]:
training_dataset_df[
    "has_any_bias"
] = (
    training_dataset_df[
        "n_biases_present"
    ] > 0
).astype(int)

In [ ]:
training_dataset_df.head()

In [ ]:
# Guardado 
MODELING_DIR = PROJECT_ROOT / "data" / "modeling"

MODELING_DIR.mkdir(
    parents=True,
    exist_ok=True
)

MODELING_PKL = MODELING_DIR / "modeling_dataset_v0_2.pkl"

training_dataset_df.to_pickle(MODELING_PKL)